# Clean the Network file
Preprocess `cross_references.tsv` according to the initial EDA.

In [11]:
import pandas as pd

In [12]:
raw_data = pd.read_csv('data/original_data/cross_references.tsv', sep='\t')
raw_data.sample(10)

,From Verse,To Verse,Votes,#www.openbible.info CC-BY 2026-09-07
298862,1Cor.10.10,Num.14.2,17,NaN
25895,Lev.11.43,Lev.11.41-Lev.11.42,3,NaN
104491,Job.22.26,Job.34.9,2,NaN
294543,Rom.14.1,1Cor.8.7-1Cor.8.13,24,NaN
215637,Hos.11.10,Jer.33.9,3,NaN
179339,Jer.13.25,Mic.3.11,2,NaN
91933,2Chr.32.19,Jer.32.30,2,NaN
241702,Matt.25.4,Ps.45.7,3,NaN
91487,2Chr.30.21,1Cor.5.7-1Cor.5.8,2,NaN
329726,Jas.1.2,1Pet.1.6-1Pet.1.8,164,NaN


In [13]:
data = raw_data.iloc[:,0:3]
display(data.shape)
display(data.sample(5))

(344756, 3)

,From Verse,To Verse,Votes
246414,Mark.7.24,Isa.23.12,2
766,Gen.3.5,Gen.3.7,4
256127,Luke.10.30,Jer.51.52,2
95559,Neh.3.17,Neh.3.16,2
123135,Ps.69.20,John.12.27,3


In [14]:
# remove negative and zero votes
data_positive = data.loc[data['Votes']>0]
display(data_positive.shape)
display(data_positive.sample(5))

(341242, 3)

,From Verse,To Verse,Votes
333452,1Pet.3.16,Heb.13.18,14
232431,Matt.5.42,Job.31.16-Job.31.20,3
316113,1Thess.1.5,Acts.2.33,2
237278,Matt.15.4,Exod.20.12,20
142115,Prov.8.31,John.13.1,5


In [15]:
def filter_func(row, max_verse):
    # identify if the from verse and to verse are from the same chapter
    chap1 = '.'.join(row['From Verse'].split('.')[:2])
    chap2 = '.'.join(row['To Verse'].split('.')[:2])
    same_chapter = chap1 == chap2

    # if the To Verse spans multiple verses:
    if '-' in row['To Verse']:
        # Check whether it spans multiple chapters
        start, end = row['To Verse'].split('-')
        start_parts = start.split('.')
        end_parts = end.split('.')

        # check whether 'To Verse' stays within one chapter
        not_multi_chapter = start_parts[:2] == end_parts[:2]
        
        # check the number of verses it spans
        start_verse = int(start_parts[2])
        end_verse = int(end_parts[2])
        verse_span = end_verse - start_verse + 1
    else:
        not_multi_chapter = True
        verse_span = 1
    return (not same_chapter) and not_multi_chapter and (verse_span <= max_verse)

In [16]:
# Apply filter
idx_filter = data_positive.apply(filter_func, max_verse=5, axis=1) # set max verse span to 5
data_filtered = data_positive.loc[idx_filter, :]
display(data_filtered.shape)
display(data_filtered.sample(10))

(312791, 3)

,From Verse,To Verse,Votes
220741,Mic.2.12,Hos.1.11,4
326172,Heb.7.6,Heb.11.17,3
306720,Gal.3.16,Col.2.19,1
83761,1Chr.17.9,Ezek.37.25,3
126188,Ps.82.5,Prov.1.29,7
62870,2Sam.4.8,1Sam.25.29,4
245982,Mark.6.21,Acts.12.2-Acts.12.4,2
8151,Gen.31.19,Josh.24.2,3
280611,Acts.11.29,2Cor.8.12-2Cor.8.14,7
139770,Ps.150.3,1Chr.15.28,2


In [17]:
# save filtered data
data_filtered.to_csv('data/data_filtered.tsv', sep='\t')

In [18]:
# drop votes column
data_filtered_no_votes = data_filtered.drop(columns=["Votes"])
data_filtered_no_votes.to_csv('data/data_filtered_no_votes.tsv', sep='\t')

# Clean the Bible file

In [20]:
bible_text_path = './data/original_data/kjv.txt'
bible = pd.read_csv(bible_text_path, sep='\t', skiprows=2, header=None)
bible.columns = ['verse', 'text']

bible_csv_path = './data/kjv.tsv'
bible.to_csv(bible_csv_path, sep='\t')